# 04 — CoAP Ingestion (aiocoap + SDK verify)

`POST coap://{host}:{port}/{org}/{project}/{collection}?key=<api_key>` -> Kafka `{org}.{project}.{collection}`.
Rules: POST-only, exactly 3 path segments, max 1024 B payload, `?key=` required.

In [ ]:
%pip install -q "nostradamus-ioto-sdk @ git+https://github.com/robotics-4-all/nostradamus-ioto-sdk.git@4f8e7fc7c28c859eefed9f7382d1b148a5c6eeb8" aiocoap python-dotenv

In [ ]:
import asyncio, json, os, random, time
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()
BASE_URL = os.getenv("CENOTOO_BASE_URL", "http://localhost:8000")
USERNAME = os.getenv("CENOTOO_USERNAME", "")
PASSWORD = os.getenv("CENOTOO_PASSWORD", "")
COAP_HOST = os.getenv("COAP_HOST", "localhost")
COAP_PORT = int(os.getenv("COAP_PORT", "5683"))
assert BASE_URL and USERNAME and PASSWORD, "SKIP: set CENOTOO_BASE_URL/USERNAME/PASSWORD"

def unique_name(p="nb"):
    return f"{p}_{time.strftime('%Y%m%d%H%M%S')}_{random.randint(1000,9999)}"

def wait_for(fn, pred, timeout=60, interval=2, desc="c"):
    import time as _t
    end = _t.monotonic() + timeout
    while _t.monotonic() < end:
        v = fn()
        try:
            if pred(v):
                return v
        except Exception:
            pass
        _t.sleep(interval)
    raise TimeoutError(desc)

In [ ]:
from nostradamus_ioto_sdk import NostradamusClient
from nostradamus_ioto_sdk.models import KeyType
admin = NostradamusClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)
project = admin.projects.create(name=unique_name("nbcoap"), description="nb04", tags=["notebook", "04"])
collection = admin.collections.create(project_id=project.project_id, name=unique_name("sensors"),
    description="nb04", tags=["notebook", "04"],
    collection_schema={"key": "K", "timestamp": "2026-01-01T00:00:00Z", "temperature": 20.0})
wk = admin.project_keys.create(project.project_id, KeyType.WRITE)
print("setup OK:", project.name, collection.name)

In [ ]:
import aiocoap
probe = unique_name("coapprobe")
body = {"key": probe, "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "temperature": 23.0, "humidity": 60}
uri = f"coap://{COAP_HOST}:{COAP_PORT}/{project.name}/{project.name}/{collection.name}?key={wk.api_key}"
print("POST", uri.replace(wk.api_key, "****"))
async def _post():
    ctx = await aiocoap.Context.create_client_context()
    req = aiocoap.Message(code=aiocoap.POST, uri=uri, payload=json.dumps(body).encode())
    resp = await ctx.request(req).response
    return resp
resp = asyncio.run(_post())
print("response:", resp.code)
assert str(resp.code) == "2.04 Changed", resp.code

In [ ]:
# Negative A: oversize payload (>1024 B) -> 4.13; Negative B: missing ?key= -> 4.xx
async def _neg():
    ctx = await aiocoap.Context.create_client_context()
    big = aiocoap.Message(code=aiocoap.POST, uri=uri, payload=b"x" * 2048)
    r1 = await ctx.request(big).response
    print("oversize ->", r1.code)
    nokey = f"coap://{COAP_HOST}:{COAP_PORT}/{project.name}/{project.name}/{collection.name}"
    r2 = await ctx.request(aiocoap.Message(code=aiocoap.POST, uri=nokey, payload=b"{}")).response
    print("missing key ->", r2.code)
asyncio.run(_neg())

In [ ]:
found = wait_for(lambda: admin.data.get(project_id=project.project_id, collection_id=collection.collection_id,
    filters=[{"property_name": "key", "operator": "eq", "property_value": probe}], limit=10),
    lambda r: len(r.data) >= 1, desc="coap record readable")
print("verify OK: coap -> kafka -> cassandra -> SDK read")

In [ ]:
import contextlib
with contextlib.suppress(Exception):
    admin.collections.delete(project.project_id, collection.collection_id)
with contextlib.suppress(Exception):
    admin.projects.delete(project.project_id)
admin.close()
print("cleanup done")